<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> Generate Tool-Use Training Data</h1>

Generate **diverse** training data for fine-tuning GPT-OSS 20B to use tools.

This notebook uses **GPT-OSS itself** (via LiteLLM) to generate varied examples across:
- **15 research domains** (ML, biology, physics, finance, etc.)
- **Multiple phrasing templates** per query type
- **3 tool types**: search_papers, get_paper_details, log_to_mlflow

**Approach:** GPT-OSS generates candidates, we filter and keep only valid examples. This creates high-quality, diverse training data through self-distillation.

**Next:** Run `04b-fine-tuning.ipynb` to train the model.

In [8]:
import os
import json
import random
import re
from pathlib import Path
from openai import OpenAI

from check_jupyter_flavor import check_flavor
check_flavor('fine-tuning')

# LiteLLM endpoint and API key (set by JupyterHub environment)
LITELLM_ENDPOINT = os.environ['LITELLM_ENDPOINT']
LITELLM_API_KEY = os.environ['LITELLM_MASTER_KEY']

# Connect to LiteLLM (for single requests)
client = OpenAI(
    base_url=LITELLM_ENDPOINT,
    api_key=LITELLM_API_KEY
)

# LiteLLM passthrough endpoint for batch requests
# Requests to /gptoss20/v1/chat/completions are forwarded unchanged to TensorRT-LLM,
# allowing the 'batch' field to pass through for true GPU batching
BATCH_ENDPOINT = f"{LITELLM_ENDPOINT}/gptoss20"

OUTPUT_PATH = Path("tool_use_training_data.json")
NUM_EXAMPLES = 2000

print(f"LiteLLM: {LITELLM_ENDPOINT}")
print(f"Batch endpoint: {BATCH_ENDPOINT}/v1/chat/completions")
print(f"Target: {NUM_EXAMPLES} examples")

✓ Running in correct environment: fine-tuning
  Fine-Tuning Lab (Unsloth, QLoRA, PEFT, TRL + ml-gpu)
LiteLLM: https://litellm.cmxela.com
Batch endpoint: https://litellm.cmxela.com/gptoss20/v1/chat/completions
Target: 2000 examples


---
## 1. Configuration

Define diverse domains and topics to prevent overfitting to specific content.

In [9]:
# Tool definitions
TOOLS = [
    {
        "name": "search_papers",
        "description": "Search vector database for papers matching a semantic query",
        "parameters": {"query": "string - what to search for"}
    },
    {
        "name": "get_paper_details",
        "description": "Get full content of a specific paper by title",
        "parameters": {"paper_title_fragment": "string - part of the paper title"}
    },
    {
        "name": "log_to_mlflow",
        "description": "Log research findings to MLflow experiment",
        "parameters": {
            "experiment_name": "string - name of experiment",
            "run_name": "string - name of this run",
            "findings_summary": "string - summary to log"
        }
    }
]

# 15 diverse domains
DOMAINS = [
    "machine learning",
    "protein folding",
    "climate modeling",
    "quantum computing",
    "autonomous vehicles",
    "natural language processing",
    "computer vision",
    "drug discovery",
    "robotics",
    "renewable energy",
    "genomics",
    "materials science",
    "financial modeling",
    "epidemiology",
    "neural architecture search"
]

# 5 topics per domain (75 unique topics total)
TOPICS = {
    "machine learning": ["transformer architectures", "attention mechanisms", "gradient descent optimization", "batch normalization", "dropout regularization"],
    "protein folding": ["alpha helix prediction", "protein-protein interactions", "molecular dynamics", "sequence alignment", "structure determination"],
    "climate modeling": ["carbon cycle simulation", "ocean temperature prediction", "atmospheric CO2 levels", "ice sheet dynamics", "weather pattern analysis"],
    "quantum computing": ["qubit error correction", "quantum entanglement", "superconducting circuits", "quantum algorithms", "decoherence mitigation"],
    "autonomous vehicles": ["lidar perception", "path planning algorithms", "sensor fusion", "pedestrian detection", "lane keeping systems"],
    "natural language processing": ["text summarization", "sentiment analysis", "named entity recognition", "machine translation", "question answering"],
    "computer vision": ["object detection", "image segmentation", "facial recognition", "pose estimation", "optical flow"],
    "drug discovery": ["molecular docking", "ADMET prediction", "virtual screening", "lead optimization", "pharmacokinetics modeling"],
    "robotics": ["motion planning", "grasp detection", "SLAM algorithms", "inverse kinematics", "reinforcement learning for control"],
    "renewable energy": ["solar cell efficiency", "wind turbine optimization", "energy storage systems", "grid integration", "photovoltaic materials"],
    "genomics": ["gene expression analysis", "variant calling", "CRISPR editing", "single-cell sequencing", "epigenetic modifications"],
    "materials science": ["crystal structure prediction", "polymer synthesis", "nanomaterial properties", "alloy design", "thin film deposition"],
    "financial modeling": ["risk assessment", "portfolio optimization", "market prediction", "credit scoring", "algorithmic trading"],
    "epidemiology": ["disease spread modeling", "vaccine effectiveness", "contact tracing", "mortality prediction", "outbreak detection"],
    "neural architecture search": ["search space design", "performance prediction", "multi-objective optimization", "hardware-aware NAS", "one-shot methods"]
}

# Query templates for search_papers
SEARCH_TEMPLATES = [
    "Find papers about {topic}",
    "Search for research on {topic}",
    "What papers exist about {topic}?",
    "Look up papers related to {topic}",
    "I need papers on {topic}",
    "Can you find research about {topic}?",
    "Search the database for {topic}",
    "What's the latest research on {topic}?",
    "Find me papers discussing {topic}",
    "Look for papers that cover {topic}"
]

print(f"Domains: {len(DOMAINS)}")
print(f"Topics per domain: {len(TOPICS[DOMAINS[0]])}")
print(f"Total unique topics: {sum(len(t) for t in TOPICS.values())}")
print(f"Search templates: {len(SEARCH_TEMPLATES)}")

Domains: 15
Topics per domain: 5
Total unique topics: 75
Search templates: 10


---
## 2. Example Generators (with GPU Batch Support)

Use **GPT-OSS itself** to generate diverse, natural training examples.

**Approach (Self-Distillation):**
- GPT-OSS generates candidate examples
- We parse and validate the output
- Only well-formed examples are kept
- Failed/malformed examples are discarded

**GPU Batching:** Uses the `batch` field in `/v1/chat/completions` to process
multiple prompts in a single GPU batch operation for 3-5x speedup.

In [10]:
def build_search_prompt(domain, topic):
    """Build prompt for search_papers example generation."""
    template = random.choice(SEARCH_TEMPLATES)
    user_query = template.format(topic=f"{topic} in {domain}")
    
    prompt = f"""Generate a natural search query for finding research papers.

User asked: "{user_query}"

Create a good semantic search query (just the query text, no extra formatting).
Make it specific to {topic} and {domain}.

Example queries:
- "transformer attention mechanisms deep learning"
- "protein folding molecular dynamics simulation"
- "quantum error correction superconducting qubits"

Your search query:"""
    
    return {"prompt": prompt, "user_query": user_query, "domain": domain, "topic": topic, "type": "search"}


def build_details_prompt(domain, topic):
    """Build prompt for get_paper_details example generation."""
    prompt = f"""Generate a natural user query asking for paper details about {topic} in {domain}.

Make it sound like a real user request. Examples:
- "Can you show me the details of that paper on transformer architectures?"
- "Get me the full paper about protein folding dynamics"
- "I need the details on the quantum computing paper"

Also suggest a paper title fragment to search for (3-5 words from a realistic paper title).

Output format:
User query: [the natural query]
Title fragment: [3-5 words]"""
    
    return {"prompt": prompt, "domain": domain, "topic": topic, "type": "details"}


def build_mlflow_prompt(domain, topic):
    """Build prompt for log_to_mlflow example generation."""
    prompt = f"""Generate a natural user request to log research findings to MLflow for {topic} in {domain}.

Examples:
- "Log my transformer training results to MLflow"
- "Save these protein folding findings to my experiment"
- "Record this analysis in MLflow under my quantum computing project"

Also generate:
- An experiment name (kebab-case, relevant to domain)
- A run name (kebab-case, relevant to topic)
- A brief findings summary (1 sentence)

Output format:
User query: [natural request]
Experiment: [experiment-name]
Run: [run-name]
Findings: [one sentence summary]"""
    
    return {"prompt": prompt, "domain": domain, "topic": topic, "type": "mlflow"}


def parse_search_response(response_text, meta):
    """Parse search example response."""
    search_query = response_text.strip().strip('"').strip("'")
    return {
        "user_query": meta["user_query"],
        "tool_call": {
            "name": "search_papers",
            "arguments": {"query": search_query}
        }
    }


def parse_details_response(response_text, meta):
    """Parse details example response."""
    content = response_text.strip()
    lines = content.split('\n')
    
    user_query = f"Get details on the {meta['topic']} paper"
    fragment = meta['topic']
    
    for line in lines:
        if line.lower().startswith('user query:'):
            user_query = line.split(':', 1)[1].strip().strip('"').strip("'")
        elif line.lower().startswith('title fragment:'):
            fragment = line.split(':', 1)[1].strip().strip('"').strip("'")
    
    return {
        "user_query": user_query,
        "tool_call": {
            "name": "get_paper_details",
            "arguments": {"paper_title_fragment": fragment}
        }
    }


def parse_mlflow_response(response_text, meta):
    """Parse mlflow example response."""
    content = response_text.strip()
    domain = meta['domain']
    topic = meta['topic']
    
    user_query = f"Log my {topic} analysis to MLflow"
    experiment_name = f"{domain.replace(' ', '-')}-research"
    run_name = f"{topic.replace(' ', '-')}-run"
    findings_summary = f"Analysis of {topic} in {domain}"
    
    for line in content.split('\n'):
        if line.lower().startswith('user query:'):
            user_query = line.split(':', 1)[1].strip().strip('"').strip("'")
        elif line.lower().startswith('experiment:'):
            experiment_name = line.split(':', 1)[1].strip()
        elif line.lower().startswith('run:'):
            run_name = line.split(':', 1)[1].strip()
        elif line.lower().startswith('findings:'):
            findings_summary = line.split(':', 1)[1].strip()
    
    return {
        "user_query": user_query,
        "tool_call": {
            "name": "log_to_mlflow",
            "arguments": {
                "experiment_name": experiment_name,
                "run_name": run_name,
                "findings_summary": findings_summary
            }
        }
    }


PARSERS = {
    "search": parse_search_response,
    "details": parse_details_response,
    "mlflow": parse_mlflow_response
}


def generate_batch(batch_metas, max_tokens=150, temperature=0.7):
    """Generate examples using GPU batch processing via LiteLLM passthrough.
    
    Args:
        batch_metas: List of dicts with 'prompt', 'type', and metadata
        max_tokens: Max tokens per response
        temperature: Sampling temperature
        
    Returns:
        List of parsed examples (None for failures)
    """
    if not batch_metas:
        return []
    
    # Build batch request
    batch_requests = [
        {"messages": [{"role": "user", "content": meta["prompt"]}]}
        for meta in batch_metas
    ]
    
    # Send batch request via LiteLLM passthrough endpoint
    import httpx
    
    response = httpx.post(
        f"{BATCH_ENDPOINT}/v1/chat/completions",
        json={
            "batch": batch_requests,
            "max_tokens": max_tokens,
            "temperature": temperature
        },
        headers={"Authorization": f"Bearer {LITELLM_API_KEY}"},
        timeout=120.0
    )
    
    if response.status_code != 200:
        raise Exception(f"Batch request failed: {response.status_code} - {response.text}")
    
    result = response.json()
    choices = result.get("choices", [])
    
    # Parse each response
    examples = []
    for i, meta in enumerate(batch_metas):
        try:
            if i < len(choices):
                content = choices[i].get("message", {}).get("content", "")
                parser = PARSERS.get(meta["type"])
                if parser and content:
                    example = parser(content, meta)
                    examples.append(example)
                else:
                    examples.append(None)
            else:
                examples.append(None)
        except Exception:
            examples.append(None)
    
    return examples


# Keep individual generators for testing/fallback (use LiteLLM)
def generate_search_example(domain, topic):
    """Generate a search_papers example (single, via LiteLLM)."""
    meta = build_search_prompt(domain, topic)
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": meta["prompt"]}],
        max_tokens=50,
        temperature=0.7
    )
    return parse_search_response(response.choices[0].message.content, meta)


def generate_details_example(domain, topic):
    """Generate a get_paper_details example (single, via LiteLLM)."""
    meta = build_details_prompt(domain, topic)
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": meta["prompt"]}],
        max_tokens=100,
        temperature=0.8
    )
    return parse_details_response(response.choices[0].message.content, meta)


def generate_mlflow_example(domain, topic):
    """Generate a log_to_mlflow example (single, via LiteLLM)."""
    meta = build_mlflow_prompt(domain, topic)
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": meta["prompt"]}],
        max_tokens=150,
        temperature=0.8
    )
    return parse_mlflow_response(response.choices[0].message.content, meta)


print("✓ Generators ready")
print(f"  - Batch mode: {BATCH_ENDPOINT}")
print(f"  - Single mode: {LITELLM_ENDPOINT}")

✓ Generators ready
  - Batch mode: https://litellm.cmxela.com/gptoss20
  - Single mode: https://litellm.cmxela.com


---
## 3. Convert to Training Format

Convert generated examples to OpenAI fine-tuning format.

Training examples contain:
1. **User message** - the query
2. **Assistant message** - the tool call (with `tool_calls` field)
3. **Tools definition** - available tools for context

The model learns to produce the correct tool call given the user query.

In [11]:
def convert_to_training_format(example):
    """Convert to proper Harmony format for GPT-OSS fine-tuning.
    
    Based on OpenAI Harmony specification:
    - Tool declarations go in 'developer' message using TypeScript namespace
    - Tool calls use 'commentary' channel with 'to=functions.toolname'
    - Format: messages array with role, and optionally channel
    
    For tool-use training, we create a complete example showing:
    1. Developer message with tool declarations
    2. User message with query  
    3. Assistant message on 'commentary' channel making tool call
    """
    if not example:
        return None
    
    tool_call = example.get("tool_call", {})
    
    if not tool_call.get("name") or not tool_call.get("arguments"):
        return None
    
    # Create tool declarations in TypeScript namespace format
    tool_declarations = """namespace functions {
  // Search for research papers in vector database
  type search_papers = (_: {
    query: string;  // semantic search query
  }) => any;
  
  // Get full details of a specific paper
  type get_paper_details = (_: {
    paper_title_fragment: string;  // part of paper title to search for
  }) => any;
  
  // Log research findings to MLflow
  type log_to_mlflow = (_: {
    experiment_name: string;  // name of MLflow experiment
    run_name: string;         // name of this run
    findings_summary: string; // summary of findings to log
  }) => any;
}"""
    
    # Format the tool call arguments as JSON
    tool_args_json = json.dumps(tool_call["arguments"])
    
    return {
        "messages": [
            {
                "role": "developer",
                "content": f"# Instructions\n\nYou are a research assistant that helps users find and analyze papers.\n\n{tool_declarations}"
            },
            {
                "role": "user",
                "content": example["user_query"]
            },
            {
                "role": "assistant",
                "channel": "commentary",
                "to": f"functions.{tool_call['name']}",
                "content": tool_args_json
            }
        ]
    }


print("✓ Converter ready (proper Harmony format with tool declarations)")

✓ Converter ready (proper Harmony format with tool declarations)


---
## 4. Generate Training Data (GPU Batch Mode)

Generate diverse examples across all domains with distribution:
- 50% search_papers
- 30% get_paper_details  
- 20% log_to_mlflow

**GPU Batching**: Sends batch requests via LiteLLM passthrough endpoint to TensorRT-LLM.
The passthrough forwards the `batch` field unchanged, enabling true GPU batching
via `llm.generate([prompts])` for 3-5x speedup over sequential calls.

In [13]:
# Check if we already have generated data
CHECKPOINT_PATH = Path("tool_use_checkpoint.json")

if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH) as f:
        examples = json.load(f)
    print(f"Loaded {len(examples)} examples from checkpoint")
    print("Delete tool_use_checkpoint.json to regenerate")
else:
    from tqdm import tqdm
    import time
    
    examples = []
    total_attempts = 0
    
    # Track detailed error info
    error_details = {
        "generation_failed": 0,
        "conversion_failed": 0,
        "exception_errors": 0
    }
    
    BATCH_SIZE = 8  # GPU batch size (matches server MAX_BATCH_SIZE)
    MAX_ATTEMPTS = NUM_EXAMPLES * 3
    
    print(f"Generating {NUM_EXAMPLES} examples using GPU batch mode...")
    print(f"Batch size: {BATCH_SIZE} (true GPU batching)")
    print(f"Maximum attempts: {MAX_ATTEMPTS}\n")
    
    def build_batch_metas(count):
        """Build batch of prompt metadata."""
        metas = []
        for _ in range(count):
            domain = random.choice(DOMAINS)
            topic = random.choice(TOPICS[domain])
            
            # Pick generator based on distribution: 50% search, 30% details, 20% mlflow
            r = random.random()
            if r < 0.5:
                metas.append(build_search_prompt(domain, topic))
            elif r < 0.8:
                metas.append(build_details_prompt(domain, topic))
            else:
                metas.append(build_mlflow_prompt(domain, topic))
        return metas
    
    # Create progress bar
    pbar = tqdm(total=NUM_EXAMPLES, desc="Generated", unit="ex")
    
    # Track tool distribution
    tool_counts = {"search_papers": 0, "get_paper_details": 0, "log_to_mlflow": 0}
    
    start_time = time.time()
    
    # Generate in GPU batches
    while len(examples) < NUM_EXAMPLES and total_attempts < MAX_ATTEMPTS:
        # Build batch
        remaining = NUM_EXAMPLES - len(examples)
        batch_size = min(BATCH_SIZE, remaining + 2)  # Small buffer for failures
        batch_metas = build_batch_metas(batch_size)
        
        try:
            # GPU batch generation - use 1024 tokens to avoid truncation
            batch_results = generate_batch(batch_metas, max_tokens=1024, temperature=0.7)
            
            # Process results
            for i, (meta, result) in enumerate(zip(batch_metas, batch_results)):
                total_attempts += 1
                
                if result:
                    training_example = convert_to_training_format(result)
                    if training_example:
                        examples.append(training_example)
                        tool_name = result["tool_call"]["name"]
                        tool_counts[tool_name] += 1
                        pbar.update(1)
                        
                        # Update progress
                        elapsed = time.time() - start_time
                        rate = len(examples) / elapsed if elapsed > 0 else 0
                        pbar.set_postfix({
                            "rate": f"{rate:.1f}/s",
                            "batch": BATCH_SIZE,
                            "errors": sum(error_details.values())
                        })
                    else:
                        error_details["conversion_failed"] += 1
                else:
                    error_details["generation_failed"] += 1
                    
                # Stop if we have enough
                if len(examples) >= NUM_EXAMPLES:
                    break
                    
        except Exception as e:
            error_details["exception_errors"] += 1
            print(f"\nBatch error: {e}")
            # Continue with next batch
        
        # Save checkpoint periodically
        if len(examples) % 50 == 0 and len(examples) > 0:
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(examples, f, indent=2)
            elapsed = time.time() - start_time
            print(f"\n💾 Checkpoint: {len(examples)} examples in {elapsed:.1f}s ({len(examples)/elapsed:.1f}/s)")
    
    pbar.close()
    
    # Final save
    with open(CHECKPOINT_PATH, 'w') as f:
        json.dump(examples, f, indent=2)
    
    elapsed = time.time() - start_time
    total_errors = sum(error_details.values())
    success_rate = (len(examples) / total_attempts * 100) if total_attempts > 0 else 0
    
    print(f"\n{'='*60}")
    print(f"GENERATION COMPLETE (GPU Batch Mode)")
    print(f"  Time elapsed: {elapsed:.1f}s")
    print(f"  Throughput: {len(examples)/elapsed:.1f} examples/sec")
    print(f"  Total attempts: {total_attempts}")
    print(f"  Successful examples: {len(examples)}")
    print(f"  Total errors: {total_errors}")
    print(f"  Success rate: {success_rate:.1f}%")
    print(f"\nTool Distribution:")
    for tool, count in tool_counts.items():
        pct = 100 * count / len(examples) if len(examples) > 0 else 0
        print(f"  - {tool}: {count} ({pct:.1f}%)")
    print(f"{'='*60}")

print(f"\n✓ Loaded {len(examples)} training examples")

Generating 2000 examples using GPU batch mode...
Batch size: 8 (true GPU batching)
Maximum attempts: 6000



Generated:  10%|█         | 200/2000 [06:26<56:40,  1.89s/ex, rate=0.5/s, batch=8, errors=0]  


💾 Checkpoint: 200 examples in 386.9s (0.5/s)


Generated:  20%|██        | 400/2000 [13:03<54:59,  2.06s/ex, rate=0.5/s, batch=8, errors=0]  


💾 Checkpoint: 400 examples in 783.9s (0.5/s)


...


💾 Checkpoint: 1000 examples in 1937.8s (0.5/s)


Generated:  60%|██████    | 1200/2000 [38:36<27:38,  2.07s/ex, rate=0.5/s, batch=8, errors=0]


💾 Checkpoint: 1200 examples in 2316.4s (0.5/s)


Generated: 100%|██████████| 2000/2000 [1:03:50<00:00,  1.92s/ex, rate=0.5/s, batch=8, errors=1]


💾 Checkpoint: 2000 examples in 3830.3s (0.5/s)

GENERATION COMPLETE (GPU Batch Mode)
  Time elapsed: 3830.4s
  Throughput: 0.5 examples/sec
  Total attempts: 2001
  Successful examples: 2000
  Total errors: 1
  Success rate: 100.0%

Tool Distribution:
  - search_papers: 993 (49.6%)
  - get_paper_details: 572 (28.6%)
  - log_to_mlflow: 435 (21.8%)

✓ Loaded 2000 training examples


---
## 5. Preview Examples

In [14]:
print("Sample training examples:\n")

for i, ex in enumerate(random.sample(examples, min(3, len(examples)))):
    msgs = ex["messages"]
    print(f"{'='*60}")
    print(f"Example {i+1}:")
    print(f"{'='*60}")
    
    # Find the developer, user, and assistant messages
    for msg in msgs:
        if msg["role"] == "developer":
            print(f"Developer: {msg['content'][:100]}...")
        elif msg["role"] == "user":
            print(f"User: {msg['content']}")
        elif msg["role"] == "assistant":
            print(f"Assistant → {msg.get('to', 'unknown')}")
            print(f"Content: {msg['content']}")
    print()

Sample training examples:

Example 1:
Developer: # Instructions

You are a research assistant that helps users find and analyze papers.

namespace fu...
User: Get details on the qubit error correction paper
Assistant → functions.get_paper_details
Content: {"paper_title_fragment": "qubit error correction"}

Example 2:
Developer: # Instructions

You are a research assistant that helps users find and analyze papers.

namespace fu...
User: Find papers about solar cell efficiency in renewable energy
Assistant → functions.search_papers
Content: {"query": "text='solar cell efficiency renewable energy research papers"}

Example 3:
Developer: # Instructions

You are a research assistant that helps users find and analyze papers.

namespace fu...
User: Get details on the gradient descent optimization paper
Assistant → functions.get_paper_details
Content: {"paper_title_fragment": "gradient descent optimization"}



---
## 6. Save for Training

Save in the format expected by `04b-fine-tuning.ipynb`.

In [15]:
# Save to output file
with open(OUTPUT_PATH, 'w') as f:
    json.dump(examples, f, indent=2)

print(f"Saved to: {OUTPUT_PATH}")
print(f"Size: {OUTPUT_PATH.stat().st_size / 1024:.1f} KB")
print(f"Examples: {len(examples)}")
print(f"\nNext: Run 04b-fine-tuning.ipynb to train the model")

Saved to: tool_use_training_data.json
Size: 2224.6 KB
Examples: 2000

Next: Run 04b-fine-tuning.ipynb to train the model
